In [1]:
#SetUp

# Dataset 1: malicious_phish.csv

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)

DATA_PATH = "dataset/raw/malicious_phish.csv"
df = pd.read_csv(DATA_PATH)
df.head(5)

,url,type
0,br-icloud.com.br,phishing
1,mp3raid.com/music/krizz_kaliko.html,benign
2,bopsecrets.org/rexroth/cr/1.htm,benign
3,http://www.garage-pirenne.be/index.php?option=...,defacement
4,http://adventure-nicaragua.net/index.php?optio...,defacement


## Shapes & dtypes

In [3]:
print(df.shape)
df.info()

# Missing Values
print(df.isna().sum())

# Duplicates
print(df.duplicated().sum())
print(df.duplicated(subset=["url"]).sum())

# Label Sanity Check 
print(df["type"].unique())
print(df["type"].value_counts())

(651191, 2)
<class 'pandas.DataFrame'>
RangeIndex: 651191 entries, 0 to 651190
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   url     651191 non-null  str  
 1   type    651191 non-null  str  
dtypes: str(2)
memory usage: 9.9 MB
url     0
type    0
dtype: int64


10066
10072
<StringArray>
['phishing', 'benign', 'defacement', 'malware']
Length: 4, dtype: str
type
benign        428103
defacement     96457
phishing       94111
malware        32520
Name: count, dtype: int64


## Deeper Quality Checks

In [4]:
#1. Garbled URLs
valid = df["url"].str.match(r'^[\x20-\x7e]+$') & (df["url"].str.len() >= 4)
print("garbled urls:", (~valid).sum())

#2. Whitespaces in URLS
print((df["url"] !=df["url"].str.strip()).sum())

#3. Conflicting labels on duplicate URLs
dupes = df[df.duplicated("url", keep=False)]
conflicts = dupes.groupby("url")["type"].nunique()
print("conflicting-labels urls:", (conflicts > 1).sum())

#4.Per-class duplicate breakdown 
print(df[df.duplicated()]["type"].value_counts())


garbled urls: 923
7
conflicting-labels urls: 6


type
malware       8875
defacement    1149
benign          23
phishing        19
Name: count, dtype: int64


## 2. Data Cleaning

In [5]:
#1.Drop Duplicate Rows
df = df.drop_duplicates()

#2.Drop Whitespaces in URLs
df["url"] = df["url"].str.strip()

#3.Drop Corrupted URLs
valid = df["url"].str.match(r'^[\x20-\x7e]+$') & (df["url"].str.len() >= 4)
print("garbled urls:", (~valid).sum())
df = df[valid]

#4.Conflicting labels on duplicate URLs
dupes = df[df.duplicated("url", keep=False)]
conflicts = dupes.groupby("url")["type"].nunique()
conflicting_urls = conflicts[conflicts > 1].index
print("conflicting-labels url removed:", df["url"].isin(conflicting_urls).sum())
df = df[~df["url"].isin(conflicting_urls)]

df.to_csv("dataset/clean/malicious_phish_cleaned.csv", index=False)
print(f"saved {df.shape[0]} rows -> dataset/clean/malicious_phish_cleaned.csv")

garbled urls: 915
conflicting-labels url removed: 12


saved 640198 rows -> dataset/clean/malicious_phish_cleaned.csv


# Dataset 3: PhiUSIIL_Phishing_URL_Dataset 2.csv

In [6]:
DATA_PATH_3 = "dataset/raw/PhiUSIIL_Phishing_URL_Dataset.csv"
df3 = pd.read_csv(DATA_PATH_3)
df3.head(5)

,FILENAME,URL,URLLength,Domain,DomainLength,IsDomainIP,TLD,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,URLCharProb,TLDLength,NoOfSubDomain,HasObfuscation,NoOfObfuscatedChar,ObfuscationRatio,NoOfLettersInURL,LetterRatioInURL,NoOfDegitsInURL,DegitRatioInURL,NoOfEqualsInURL,NoOfQMarkInURL,NoOfAmpersandInURL,NoOfOtherSpecialCharsInURL,SpacialCharRatioInURL,IsHTTPS,LineOfCode,LargestLineLength,HasTitle,Title,DomainTitleMatchScore,URLTitleMatchScore,HasFavicon,Robots,IsResponsive,NoOfURLRedirect,NoOfSelfRedirect,HasDescription,NoOfPopup,NoOfiFrame,HasExternalFormSubmit,HasSocialNet,HasSubmitButton,HasHiddenFields,HasPasswordField,Bank,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label
0,521848.txt,https://www.southbankmosaics.com,31,www.southbankmosaics.com,24,0,com,100.0,1.000000,0.522907,0.061933,3,1,0,0,0.0,18,0.581,0,0.0,0,0,0,1,0.032,1,558,9381,1,à¸‚à¹ˆà¸²à¸§à¸ªà¸” à¸‚à¹ˆà¸²à¸§à¸§à¸±à¸™à¸™à¸µ...,0.000000,0.000000,0,1,1,0,0,0,0,1,0,0,1,1,0,1,0,0,1,34,20,28,119,0,124,1
1,31372.txt,https://www.uni-mainz.de,23,www.uni-mainz.de,16,0,de,100.0,0.666667,0.032650,0.050207,2,1,0,0,0.0,9,0.391,0,0.0,0,0,0,2,0.087,1,618,9381,1,johannes gutenberg-universitÃ¤t mainz,55.555556,55.555556,1,1,0,0,0,0,0,0,0,1,1,0,0,0,0,0,1,50,9,8,39,0,217,1
2,597387.txt,https://www.voicefmradio.co.uk,29,www.voicefmradio.co.uk,22,0,uk,100.0,0.866667,0.028555,0.064129,2,2,0,0,0.0,15,0.517,0,0.0,0,0,0,2,0.069,1,467,682,1,voice fm southampton,46.666667,46.666667,0,1,1,0,0,1,0,0,0,0,1,1,0,0,0,0,1,10,2,7,42,2,5,1
3,554095.txt,https://www.sfnmjournal.com,26,www.sfnmjournal.com,19,0,com,100.0,1.000000,0.522907,0.057606,3,1,0,0,0.0,13,0.500,0,0.0,0,0,0,1,0.038,1,6356,26824,1,home page: seminars in fetal and neonatal medi...,0.000000,0.000000,0,1,1,0,0,0,1,12,0,1,1,1,0,0,1,1,1,3,27,15,22,1,31,1
4,151578.txt,https://www.rewildingargentina.org,33,www.rewildingargentina.org,26,0,org,100.0,1.000000,0.079963,0.059441,3,1,0,0,0.0,20,0.606,0,0.0,0,0,0,1,0.030,1,6089,28404,1,fundaciÃ³n rewilding argentina,100.000000,100.000000,0,1,1,1,1,1,0,2,0,1,1,1,0,1,1,0,1,244,15,34,72,1,85,1


## Shapes & dtypes

In [7]:
print(df3.shape)
df3.info()

# Missing Values
print(df3.isna().sum())

# Duplicates
print(df3.duplicated().sum())

# Label Sanity Check
print(df3["label"].unique())
print(df3["label"].value_counts())

(235795, 56)
<class 'pandas.DataFrame'>
RangeIndex: 235795 entries, 0 to 235794
Data columns (total 56 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   FILENAME                    235795 non-null  str    
 1   URL                         235795 non-null  str    
 2   URLLength                   235795 non-null  int64  
 3   Domain                      235795 non-null  str    
 4   DomainLength                235795 non-null  int64  
 5   IsDomainIP                  235795 non-null  int64  
 6   TLD                         235795 non-null  str    
 7   URLSimilarityIndex          235795 non-null  float64
 8   CharContinuationRate        235795 non-null  float64
 9   TLDLegitimateProb           235795 non-null  float64
 10  URLCharProb                 235795 non-null  float64
 11  TLDLength                   235795 non-null  int64  
 12  NoOfSubDomain               235795 non-null  int64  
 13  HasObfuscati

0
[1 0]
label
1    134850
0    100945
Name: count, dtype: int64


## Deeper Quality Checks

In [8]:
#1. Non-informative ID column
print("FILENAME unique vs rows:", df3["FILENAME"].nunique(), len(df3))

#2. Garbled/non-ASCII URLs
valid = df3["URL"].str.match(r'^[\x20-\x7e]+$')
print("garbled urls:", (~valid).sum())

#3. Duplicate URLs
print("duplicate urls:", df3.duplicated("URL").sum())

#4. Conflicting labels on duplicate URLs
dupes = df3[df3.duplicated("URL", keep=False)]
conflicts = dupes.groupby("URL")["label"].nunique()
print("conflicting-label urls:", (conflicts > 1).sum())

#5. Ratio/index/probability columns out of expected bounds
ratio_cols = [c for c in df3.columns if "Ratio" in c or "Prob" in c or "Score" in c or "Index" in c]
print(df3[ratio_cols].describe().T[["min", "max"]])

FILENAME unique vs rows: 235795 235795
garbled urls: 2
duplicate urls: 425
conflicting-label urls: 0
                            min         max
URLSimilarityIndex     0.155574  100.000000
TLDLegitimateProb      0.000000    0.522907
URLCharProb            0.001083    0.090824
ObfuscationRatio       0.000000    0.348000
LetterRatioInURL       0.000000    0.926000
DegitRatioInURL        0.000000    0.684000
SpacialCharRatioInURL  0.000000    0.397000
DomainTitleMatchScore  0.000000  100.000000
URLTitleMatchScore     0.000000  100.000000


## 2. Data Cleaning

In [9]:
#1. Drop the non-informative row-ID column
df3 = df3.drop(columns=["FILENAME"])

#2. Drop garbled/non-ASCII URLs
valid = df3["URL"].str.match(r'^[\x20-\x7e]+$')
print("garbled urls removed:", (~valid).sum())
df3 = df3[valid]

#3. Drop duplicate URLs (keep first occurrence - no label conflicts, so safe)
print("duplicate url rows removed:", df3.duplicated("URL").sum())
df3 = df3.drop_duplicates(subset="URL")

#4. Label balance check (informational - handle with stratified split / class weighting downstream, not by dropping rows)
print(df3["label"].value_counts(normalize=True))

#5. Keep only URL and label - the other ~50 columns are PhiUSIIL's own
#   precomputed features, not raw data, and don't belong in a merged corpus
df3 = df3[["URL", "label"]]

#6. Flip label: PhiUSIIL uses 1=legitimate, this project uses 1=malicious/phishing
df3["label"] = 1 - df3["label"]
print(df3["label"].value_counts(normalize=True))

garbled urls removed: 2
duplicate url rows removed: 425
label
1    0.572933
0    0.427067
Name: proportion, dtype: float64
label
0    0.572933
1    0.427067
Name: proportion, dtype: float64


In [10]:
df3.to_csv("dataset/clean/phiusiil_phishing_url_dataset_cleaned.csv", index=False)

# Dataset 4: urldata.csv

In [11]:
DATA_PATH_4 = "dataset/raw/urldata.csv"
df4 = pd.read_csv(DATA_PATH_4)
df4.head(5)

,Unnamed: 0,url,label,result
0,0,https://www.google.com,benign,0
1,1,https://www.youtube.com,benign,0
2,2,https://www.facebook.com,benign,0
3,3,https://www.baidu.com,benign,0
4,4,https://www.wikipedia.org,benign,0


## Shapes & dtypes

In [12]:
print(df4.shape)
df4.info()

# Missing Values
print(df4.isna().sum())

# Duplicates
print(df4.duplicated().sum())
print(df4.duplicated(subset=["url"]).sum())

# Label Sanity Check
print(df4["label"].unique())
print(df4["label"].value_counts())

(450176, 4)
<class 'pandas.DataFrame'>
RangeIndex: 450176 entries, 0 to 450175
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype
---  ------      --------------   -----
 0   Unnamed: 0  450176 non-null  int64
 1   url         450176 non-null  str  
 2   label       450176 non-null  str  
 3   result      450176 non-null  int64
dtypes: int64(2), str(2)
memory usage: 13.7 MB
Unnamed: 0    0
url           0
label         0
result        0
dtype: int64
0


0
<StringArray>
['benign', 'malicious']
Length: 2, dtype: str
label
benign       345738
malicious    104438
Name: count, dtype: int64


## Deeper Quality Check

In [13]:
#1. Non-informative ID column (checked here, dropped later in Data Cleaning)
print("Unnamed: 0 looks like a row index:", (df4["Unnamed: 0"] == df4.index).all())

#2. Drop "result" - redundant numeric encoding of "label"
print(pd.crosstab(df4["label"], df4["result"]))

#3. Garbled/non-ASCII URLs
valid = df4["url"].str.match(r'^[\x20-\x7e]+$')
print("garbled urls:", (~valid).sum())

#4. Whitespace in URLs
print((df4["url"] != df4["url"].str.strip()).sum())

#5. Duplicate URLs
print("duplicate urls:", df4.duplicated("url").sum())

#6. Conflicting labels on duplicate URLs
dupes = df4[df4.duplicated("url", keep=False)]
conflicts = dupes.groupby("url")["label"].nunique()
print("conflicting-label urls:", (conflicts > 1).sum())

Unnamed: 0 looks like a row index: True
result          0       1
label                    
benign     345738       0
malicious       0  104438


garbled urls: 312


0
duplicate urls: 0
conflicting-label urls: 0


## Data Cleaning

In [14]:
#1. Drop the non-informative row-ID column
df4 = df4.drop(columns=["Unnamed: 0"])

#2. Drop "result" - redundant numeric encoding of "label" (see Deeper Quality Checks)
df4 = df4.drop(columns=["result"])

#3. Drop Whitespaces in URLs
df4["url"] = df4["url"].str.strip()

#4. Drop Corrupted URLs
valid = df4["url"].str.match(r'^[\x20-\x7e]+$')
print("garbled urls removed:", (~valid).sum())
df4 = df4[valid]

#5. Drop Duplicate Rows
print("duplicate rows removed:", df4.duplicated().sum())
df4 = df4.drop_duplicates()

#6. Conflicting labels on duplicate URLs
dupes = df4[df4.duplicated("url", keep=False)]
conflicts = dupes.groupby("url")["label"].nunique()
conflicting_urls = conflicts[conflicts > 1].index
print("conflicting-labels url removed:", df4["url"].isin(conflicting_urls).sum())
df4 = df4[~df4["url"].isin(conflicting_urls)]

# Label balance check
print(df4["label"].value_counts(normalize=True))

df4.to_csv("dataset/clean/urldata_cleaned.csv", index=False)
print(f"saved {df4.shape[0]} rows -> dataset/clean/urldata_cleaned.csv")

garbled urls removed: 312
duplicate rows removed: 0


conflicting-labels url removed: 0
label
benign       0.767928
malicious    0.232072
Name: proportion, dtype: float64


saved 449864 rows -> dataset/clean/urldata_cleaned.csv
